# Runbook 5.3.c: Integrity and authenticity (HMAC)  (MODEL SOLUTION, instructor copy, do not distribute)

**Skill:** Mitigate Risk · **Maps to:** 5.3.B, 5.3.C · **Language:** Python 3 (pytest)

**Steps:** 1 Brief · 2 Plan · 3 Build · 4 Test · 5 Try it on the evidence · 6 Wrap up

## 1. Brief

**The situation.** An API needs to verify that requests were sent by a client holding a shared secret key, and
that the request body wasn't tampered with in transit. Rather than calling Python's built-in
`hmac` module, build the FIPS 198-1 HMAC construction yourself on top of `hashlib.sha256`, so
you can see exactly how the ipad/opad key-nesting defeats length-extension attacks.

**Lesson objective.** Explain why a plain hash cannot prove who sent a message, describe how HMAC combines a secret key with a hash function to provide both integrity and authenticity, and classify a given integrity-check design as secure or vulnerable to tampering.

- [ ] Read the situation and say in one sentence what the tool must decide
- [ ] Find the input and the output above

## 2. Plan

Work it out on paper before you write code.

### Worked examples
```
hmac_sha256(b"secretkey", b"transfer $100 to acct 42")
  -> "5b1f9a2c...b3" (64 hex chars; must match Python's hmac.new(key, msg, sha256).hexdigest())

verify(b"secretkey", b"transfer $100 to acct 42", "5b1f9a2c...b3") -> True
verify(b"secretkey", b"transfer $900 to acct 42", "5b1f9a2c...b3") -> False  # tampered message
```

- [ ] Work the first example by hand and get the same answer

### Rules to respect
- [ ] Deterministic, offline, standard library only (`hashlib` only, no `hmac` module).
- [ ] Correctness is checked against Python's own `hmac` module inside the tests (used there only
- [ ] Must pass all cases in the tests (visible + hidden), including empty keys/messages, keys
- [ ] Write the steps your function will take, in plain words
- [ ] List two edge cases (an empty input, a tricky value)

<details><summary>Reference: the exact contract</summary>

Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

#### Entry points
```python
def hmac_sha256(key: bytes, message: bytes) -> str
def verify(key: bytes, message: bytes, tag: str) -> bool
def solve(data: dict) -> dict
```

`solve()` is the blackbox entry point; `hmac_sha256()` and `verify()` are the underlying
primitives it's built on and are tested directly as well.

#### Input
- `hmac_sha256(key, message)`: `key` and `message` are raw `bytes` of any length (including
  empty and longer-than-block-size keys).
- `verify(key, message, tag)`: same `key`/`message` as above; `tag` is a hex string (may be
  upper or lower case) claiming to be the correct HMAC-SHA256 tag.
- `solve(data)`: `data` is a `dict` with `"op"` equal to `"tag"` or `"verify"`.
  - `"tag"`: `{"op": "tag", "key": <hex str>, "message": <str>}`
  - `"verify"`: `{"op": "verify", "key": <hex str>, "message": <str>, "tag": <hex str>}`
  - `key` arrives as a hex string (decode with `bytes.fromhex`); `message` arrives as a
    plain UTF-8 string (encode with `.encode("utf-8")`).

#### Output (`result`)
- `hmac_sha256(...)` returns a lowercase hex string (64 hex characters for SHA-256).
- `verify(...)` returns `bool`.
- `solve(...)` returns `{"tag": <hex str>}` for op `"tag"`, or `{"valid": <bool>}` for op `"verify"`.

#### Guarantees / rules
- Pure functions, deterministic, no I/O, no network. Standard library only
  (`hashlib.sha256` may be used as the inner hash primitive `H()`; the ipad/opad key
  derivation and double-nesting must be implemented directly, not via the `hmac` module).
- Must implement the FIPS 198-1 HMAC construction:
  `HMAC(key, message) = H((key XOR opad) || H((key XOR ipad) || message))`, with
  `ipad = 0x36` and `opad = 0x5c` repeated to the 64-byte SHA-256 block size, and keys
  longer than the block size first hashed down with `H()` before padding.
- Any single-bit change to `message` or `key` must produce a different tag.
- `verify()` must accept tags regardless of hex case (upper/lower).

#### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).

</details>

## 3. Build it

This is the model solution.

- [ ] `solve` handles the worked examples
- [ ] `solve` handles your edge cases

In [ ]:
import hashlib

BLOCK_SIZE = 64
IPAD = 0x36
OPAD = 0x5C


def _h(data: bytes) -> bytes:
    return hashlib.sha256(data).digest()


def _derive_key(key: bytes) -> bytes:
    if len(key) > BLOCK_SIZE:
        key = _h(key)
    return key + b"\x00" * (BLOCK_SIZE - len(key))


def _xor_bytes(data: bytes, pad: int) -> bytes:
    return bytes(b ^ pad for b in data)


def hmac_sha256(key: bytes, message: bytes) -> str:
    k0 = _derive_key(key)
    inner_key = _xor_bytes(k0, IPAD)
    outer_key = _xor_bytes(k0, OPAD)
    inner_hash = _h(inner_key + message)
    tag = _h(outer_key + inner_hash)
    return tag.hex()


def _constant_time_equal(a: str, b: str) -> bool:
    if len(a) != len(b):
        return False
    result = 0
    for ca, cb in zip(a, b):
        result |= ord(ca) ^ ord(cb)
    return result == 0


def verify(key: bytes, message: bytes, tag: str) -> bool:
    expected = hmac_sha256(key, message)
    return _constant_time_equal(expected, tag.lower())


def solve(data: dict) -> dict:
    op = data["op"]
    key = bytes.fromhex(data["key"])
    message = data["message"].encode("utf-8")
    if op == "tag":
        return {"tag": hmac_sha256(key, message)}
    elif op == "verify":
        return {"valid": verify(key, message, data["tag"])}
    else:
        raise ValueError(f"unknown op: {op!r}")


## 4. Test it

The tests are the acceptance check, like a change review before a fix goes live.

- [ ] Run the tests
- [ ] Every test says PASS (if not, read the message, go back to step 3)

In [ ]:
import __main__ as impl

import hashlib
import hmac as _stdlib_hmac


def _oracle_tag(key: bytes, message: bytes) -> str:
    return _stdlib_hmac.new(key, message, hashlib.sha256).hexdigest()


def test_matches_stdlib_hmac_short_key():
    key = b"secretkey"
    message = b"transfer $100 to acct 42"
    got = impl.hmac_sha256(key, message)
    assert got == _oracle_tag(key, message)


def test_matches_stdlib_hmac_long_key():
    key = b"x" * 100
    message = b"hello world"
    got = impl.hmac_sha256(key, message)
    assert got == _oracle_tag(key, message)


def test_tampered_message_changes_tag():
    key = b"secretkey"
    original = b"transfer $100 to acct 42"
    tampered = b"transfer $900 to acct 42"
    tag_original = impl.hmac_sha256(key, original)
    tag_tampered = impl.hmac_sha256(key, tampered)
    assert tag_original != tag_tampered


def test_verify_accepts_correct_tag():
    key = b"apikey123"
    message = b"GET /accounts/42"
    tag = impl.hmac_sha256(key, message)
    assert impl.verify(key, message, tag) is True


def test_verify_rejects_tampered_message():
    key = b"apikey123"
    message = b"GET /accounts/42"
    tag = impl.hmac_sha256(key, message)
    assert impl.verify(key, b"GET /accounts/99", tag) is False


def test_solve_tag_op():
    result = impl.solve({"op": "tag", "key": "deadbeef", "message": "hello"})
    expected = _oracle_tag(bytes.fromhex("deadbeef"), b"hello")
    assert result == {"tag": expected}


def test_solve_verify_op():
    key_hex = "deadbeef"
    tag = _oracle_tag(bytes.fromhex(key_hex), b"hello")
    result = impl.solve({"op": "verify", "key": key_hex, "message": "hello", "tag": tag})
    assert result == {"valid": True}


def test_wrong_key_fails_verify():
    message = b"payload"
    tag = impl.hmac_sha256(b"key-a", message)
    assert impl.verify(b"key-b", message, tag) is False


def test_empty_message():
    key = b"secretkey"
    got = impl.hmac_sha256(key, b"")
    assert got == _oracle_tag(key, b"")


def test_empty_key():
    message = b"some data"
    got = impl.hmac_sha256(b"", message)
    assert got == _oracle_tag(b"", message)


def test_key_exactly_block_size():
    key = b"k" * 64
    message = b"boundary case"
    got = impl.hmac_sha256(key, message)
    assert got == _oracle_tag(key, message)


def test_single_bit_flip_changes_tag_completely():
    key = b"secretkey"
    a = b"password reset for user 1000"
    b = b"password reset for user 1001"
    tag_a = impl.hmac_sha256(key, a)
    tag_b = impl.hmac_sha256(key, b)
    assert tag_a != tag_b
    diff = sum(1 for x, y in zip(tag_a, tag_b) if x != y)
    assert diff > len(tag_a) // 4


def test_verify_case_insensitive_hex_tag():
    key = b"secretkey"
    message = b"case check"
    tag = impl.hmac_sha256(key, message)
    assert impl.verify(key, message, tag.upper()) is True


def test_solve_verify_op_rejects_bad_tag():
    result = impl.solve({"op": "verify", "key": "aa", "message": "x", "tag": "0" * 64})
    assert result == {"valid": False}


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")


## 5. Try it on the evidence

Call `solve` on an input you make up, and predict the answer first.

- [ ] Your prediction matched

## 6. Wrap up

- [ ] Explain in one sentence why your rule catches the problem
- [ ] Name one way an attacker could slip past it (a false negative)
- [ ] Name one harmless input it might wrongly flag (a false positive)
- [ ] Say where this check would run in a real organisation, and who would act on it